In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# إنشاء إضافة للمُحوِّل

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';





<details>
<summary><b>إصدارات الحزم</b></summary>

تم تطوير الكود الموجود في هذه الصفحة باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو ما هو أحدث منها.

```
qiskit[all]~=2.3.0
```
</details>

إنشاء [إضافة للمُحوِّل](transpiler-plugins) هو طريقة رائعة لمشاركة كود التحويل الخاص بك مع مجتمع Qiskit الأوسع، مما يتيح للمستخدمين الآخرين الاستفادة من الوظائف التي طوّرتها. شكرًا لاهتمامك بالمساهمة في مجتمع Qiskit!

قبل إنشاء إضافة للمُحوِّل، تحتاج إلى تحديد نوع الإضافة المناسب لحالتك. هناك ثلاثة أنواع من إضافات المُحوِّل:
- [**إضافة مرحلة المُحوِّل**](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins): اختر هذا النوع إذا كنت تُعرِّف مدير مسارات يمكن إحلاله محل إحدى [المراحل الست](transpiler-stages) لمدير المسارات المُهيَّأ مسبقًا.
- [**إضافة تركيب المصفوفة الأحادية**](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin): اختر هذا النوع إذا كان كود التحويل الخاص بك يأخذ كمدخل مصفوفة أحادية (ممثَّلة كمصفوفة Numpy) ويُخرج وصفًا لدائرة كمومية تُنفِّذ تلك المصفوفة الأحادية.
- [**إضافة التركيب عالي المستوى**](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin): اختر هذا النوع إذا كان كود التحويل الخاص بك يأخذ كمدخل "كائنًا عالي المستوى" مثل عامل Clifford أو دالة خطية ويُخرج وصفًا لدائرة كمومية تُنفِّذ ذلك الكائن عالي المستوى. تُمثَّل الكائنات عالية المستوى بواسطة فئات فرعية من فئة [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation).

بعد تحديد نوع الإضافة التي تريد إنشاءها، اتبع هذه الخطوات لإنشائها:

1. أنشئ فئة فرعية من فئة الإضافة المجردة المناسبة:
   - [PassManagerStagePlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin) لإضافة مرحلة المُحوِّل،
   - [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) لإضافة تركيب المصفوفة الأحادية، و
   - [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) لإضافة التركيب عالي المستوى.
2. اعرض الفئة كـ[نقطة دخول setuptools](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) في بيانات تعريف الحزمة، وعادةً ما يكون ذلك بتعديل ملف `pyproject.toml` أو `setup.cfg` أو `setup.py` الخاص بحزمتك في Python.

لا يوجد حد لعدد الإضافات التي يمكن أن تُعرِّفها حزمة واحدة، لكن يجب أن يكون لكل إضافة اسم فريد. يتضمن Qiskit SDK نفسه عددًا من الإضافات التي تحتل أسماءها محجوزة. الأسماء المحجوزة هي:

- إضافات مرحلة المُحوِّل: انظر [هذا الجدول](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#plugin-stages).
- إضافات تركيب المصفوفة الأحادية: `default`، `aqc`، `sk`
- إضافات التركيب عالي المستوى:

| فئة العملية | اسم العملية | الأسماء المحجوزة |
| --- | --- | --- |
| [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford#clifford) | `clifford` | `default`، `ag`، `bm`، `greedy`، `layers`، `lnn` |
| [LinearFunction](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction#linearfunction) | `linear_function` | `default`، `kms`، `pmh` |
| [PermutationGate](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.PermutationGate#permutationgate) | `permutation` | `default`، `kms`، `basic`، `acg`، `token_swapper` |

في الأقسام التالية، نُقدِّم أمثلة على هذه الخطوات لأنواع الإضافات المختلفة. في هذه الأمثلة، نفترض أننا نُنشئ حزمة Python تُسمَّى `my_qiskit_plugin`. للاطلاع على معلومات حول إنشاء حزم Python، يمكنك الاطلاع على [هذا البرنامج التعليمي](https://packaging.python.org/en/latest/tutorials/packaging-projects/) من موقع Python.

## مثال: إنشاء إضافة لمرحلة المُحوِّل
في هذا المثال، نُنشئ إضافة لمرحلة المُحوِّل الخاصة بمرحلة `layout` (انظر [مراحل المُحوِّل](transpiler-stages) للاطلاع على وصف المراحل الست لخط أنابيب التحويل المدمج في Qiskit).
تقوم إضافتنا ببساطة بتشغيل [VF2Layout](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.VF2Layout) لعدد من المحاولات يعتمد على مستوى التحسين المطلوب.

أولًا، نُنشئ فئة فرعية من [PassManagerStagePlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin). هناك طريقة واحدة نحتاج إلى تنفيذها تُسمَّى [`pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin#pass_manager). تأخذ هذه الطريقة كمدخل [PassManagerConfig](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManagerConfig) وتُعيد مدير المسارات الذي نُعرِّفه. يخزِّن كائن PassManagerConfig معلومات حول الـ Backend المستهدف، مثل خريطة الاقتران وبوابات الأساس.

In [1]:
# This import is needed for python versions prior to 3.10
from __future__ import annotations

from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import VF2Layout
from qiskit.transpiler.passmanager_config import PassManagerConfig
from qiskit.transpiler.preset_passmanagers import common
from qiskit.transpiler.preset_passmanagers.plugin import (
    PassManagerStagePlugin,
)


class MyLayoutPlugin(PassManagerStagePlugin):
    def pass_manager(
        self,
        pass_manager_config: PassManagerConfig,
        optimization_level: int | None = None,
    ) -> PassManager:
        layout_pm = PassManager(
            [
                VF2Layout(
                    coupling_map=pass_manager_config.coupling_map,
                    properties=pass_manager_config.backend_properties,
                    max_trials=optimization_level * 10 + 1,
                    target=pass_manager_config.target,
                )
            ]
        )
        layout_pm += common.generate_embed_passmanager(
            pass_manager_config.coupling_map
        )
        return layout_pm

الآن، نعرض الإضافة بإضافة نقطة دخول في بيانات تعريف حزمة Python الخاصة بنا.
نفترض هنا أن الفئة التي عرَّفناها مكشوفة في وحدة تُسمَّى `my_qiskit_plugin`، على سبيل المثال من خلال استيرادها في ملف `__init__.py` الخاص بوحدة `my_qiskit_plugin`.
نُعدِّل ملف `pyproject.toml` أو `setup.cfg` أو `setup.py` الخاص بحزمتنا (بحسب نوع الملف الذي اخترته لتخزين بيانات تعريف مشروع Python الخاص بك):

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.transpiler.layout"]
    "my_layout" = "my_qiskit_plugin:MyLayoutPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.transpiler.layout =
        my_layout = my_qiskit_plugin:MyLayoutPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.transpiler.layout': [
                'my_layout = my_qiskit_plugin:MyLayoutPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

انظر [جدول مراحل إضافة المُحوِّل](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#stage-table) للاطلاع على نقاط الدخول والمتطلبات الخاصة بكل مرحلة من مراحل المُحوِّل.

للتحقق من أن Qiskit يكتشف إضافتك بنجاح، ثبِّت حزمة الإضافة الخاصة بك واتبع التعليمات الموجودة في [إضافات المُحوِّل](transpiler-plugins#list-available-transpiler-stage-plugins) لسرد الإضافات المثبَّتة، وتأكد من ظهور إضافتك في القائمة:

In [2]:
from qiskit.transpiler.preset_passmanagers.plugin import list_stage_plugins

list_stage_plugins("layout")

['default', 'dense', 'sabre', 'trivial']

If our example plugin were installed, then the name `my_layout` would appear in this list.

If you want to use a built-in transpiler stage as the starting point for your transpiler stage plugin, you can obtain the pass manager for a built-in transpiler stage using [PassManagerStagePluginManager](/docs/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePluginManager#passmanagerstagepluginmanager). The following code cell shows how to do this to obtain the built-in optimization stage for optimization level 3.

In [3]:
from qiskit.transpiler.preset_passmanagers.plugin import (
    PassManagerStagePluginManager,
)

# Initialize the plugin manager
plugin_manager = PassManagerStagePluginManager()

# Here we create a pass manager config to use as an example.
# Instead, you should use the pass manager config that you already received as input
# to the pass_manager method of your PassManagerStagePlugin.
pass_manager_config = PassManagerConfig()

# Obtain the desired built-in transpiler stage
optimization = plugin_manager.get_passmanager_stage(
    "optimization", "default", pass_manager_config, optimization_level=3
)

لو كانت إضافتنا في المثال مثبَّتة، لظهر الاسم `my_layout` في هذه القائمة.

إذا أردت استخدام مرحلة مُحوِّل مدمجة كنقطة بداية لإضافة مرحلة المُحوِّل الخاصة بك، يمكنك الحصول على مدير المسارات لمرحلة مُحوِّل مدمجة باستخدام [PassManagerStagePluginManager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePluginManager#passmanagerstagepluginmanager). تُوضِّح خلية الكود التالية كيفية القيام بذلك للحصول على مرحلة التحسين المدمجة لمستوى التحسين 3.

In [4]:
import numpy as np
from qiskit.circuit import QuantumCircuit, QuantumRegister
from qiskit.converters import circuit_to_dag
from qiskit.dagcircuit.dagcircuit import DAGCircuit
from qiskit.quantum_info import Operator
from qiskit.transpiler.passes import UnitarySynthesis
from qiskit.transpiler.passes.synthesis.plugin import UnitarySynthesisPlugin


class MyUnitarySynthesisPlugin(UnitarySynthesisPlugin):
    @property
    def supports_basis_gates(self):
        # Returns True if the plugin can target a list of basis gates
        return True

    @property
    def supports_coupling_map(self):
        # Returns True if the plugin can synthesize for a given coupling map
        return False

    @property
    def supports_natural_direction(self):
        # Returns True if the plugin supports a toggle for considering
        # directionality of 2-qubit gates
        return False

    @property
    def supports_pulse_optimize(self):
        # Returns True if the plugin can optimize pulses during synthesis
        return False

    @property
    def supports_gate_lengths(self):
        # Returns True if the plugin can accept information about gate lengths
        return False

    @property
    def supports_gate_errors(self):
        # Returns True if the plugin can accept information about gate errors
        return False

    @property
    def supports_gate_lengths_by_qubit(self):
        # Returns True if the plugin can accept information about gate lengths
        # (The format of the input differs from supports_gate_lengths)
        return False

    @property
    def supports_gate_errors_by_qubit(self):
        # Returns True if the plugin can accept information about gate errors
        # (The format of the input differs from supports_gate_errors)
        return False

    @property
    def min_qubits(self):
        # Returns the minimum number of qubits the plugin supports
        return None

    @property
    def max_qubits(self):
        # Returns the maximum number of qubits the plugin supports
        return None

    @property
    def supported_bases(self):
        # Returns a dictionary of supported bases for synthesis
        return None

    def run(self, unitary: np.ndarray, **options) -> DAGCircuit:
        basis_gates = options["basis_gates"]
        synth_pass = UnitarySynthesis(basis_gates, min_qubits=3)
        qubits = QuantumRegister(3)
        circuit = QuantumCircuit(qubits)
        circuit.append(Operator(unitary).to_instruction(), qubits)
        dag_circuit = synth_pass.run(circuit_to_dag(circuit))
        return dag_circuit

## مثال: إنشاء إضافة لتوليف الوحدانيات
في هذا المثال، سننشئ إضافة لتوليف الوحدانيات تستخدم ببساطة تمريرة التحويل المدمجة [UnitarySynthesis](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.UnitarySynthesis#unitarysynthesis) لتوليف Gate. بالطبع، إضافتك الخاصة ستفعل شيئاً أكثر إثارةً من ذلك.

تُعرِّف فئة [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) الواجهة والعقد الخاص بإضافات توليف الوحدانيات. الطريقة الرئيسية هي
[`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin#run)،
التي تأخذ كمدخل مصفوفة Numpy تُخزّن مصفوفة وحدانية
وتُعيد [DAGCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.dagcircuit.DAGCircuit) يُمثّل الدائرة المُولَّفة من تلك المصفوفة الوحدانية.
إلى جانب طريقة `run`، ثمّة عدد من طرق الخصائص التي يجب تعريفها.
راجع [UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) للاطلاع على توثيق جميع الخصائص المطلوبة.

لنُنشئ الآن فئتنا الفرعية من UnitarySynthesisPlugin:

In [5]:
from qiskit.transpiler.passes.synthesis import unitary_synthesis_plugin_names

unitary_synthesis_plugin_names()

['aqc', 'clifford', 'default', 'gridsynth', 'sk']

إذا وجدت أن المدخلات المتاحة لطريقة [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin#run)
غير كافية لأغراضك، يُرجى [فتح تذكرة](https://github.com/Qiskit/qiskit/issues/new/choose) توضّح فيها متطلباتك. التغييرات على واجهة الإضافة، كإضافة مدخلات اختيارية جديدة، ستُجرى بطريقة متوافقة مع الإصدارات السابقة بحيث لا تستلزم تعديلات من الإضافات الموجودة.

> **Note:** جميع الطرق التي تبدأ بـ `supports_` محجوزة في الفئات المشتقة من `UnitarySynthesisPlugin` كجزء من الواجهة. لا تُعرِّف أي طرق `supports_*` مخصصة في فئة فرعية ما لم تكن مُعرَّفة في الفئة المجردة.

الآن، سنكشف عن الإضافة بإضافة نقطة دخول في بيانات تعريف حزمة Python الخاصة بنا.
نفترض هنا أن الفئة التي عرّفناها مكشوفة في وحدة تسمى `my_qiskit_plugin`، على سبيل المثال عبر استيرادها في ملف `__init__.py` الخاص بوحدة `my_qiskit_plugin`.
نُعدِّل ملف `pyproject.toml` أو `setup.cfg` أو `setup.py` في حزمتنا:

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.unitary_synthesis"]
    "my_unitary_synthesis" = "my_qiskit_plugin:MyUnitarySynthesisPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.unitary_synthesis =
        my_unitary_synthesis = my_qiskit_plugin:MyUnitarySynthesisPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.unitary_synthesis': [
                'my_unitary_synthesis = my_qiskit_plugin:MyUnitarySynthesisPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

كما في السابق، إذا كان مشروعك يستخدم `setup.cfg` أو `setup.py` بدلاً من `pyproject.toml`، راجع [توثيق setuptools](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) لمعرفة كيفية تكييف هذه الأسطر لحالتك.

للتحقق من أن إضافتك اكتُشفت بنجاح بواسطة Qiskit، ثبِّت حزمة الإضافة واتبع التعليمات الموجودة في [إضافات Transpiler](transpiler-plugins#list-available-unitary-synthesis-plugins) لسرد الإضافات المثبتة، وتأكد من ظهور إضافتك في القائمة:

In [6]:
from qiskit.synthesis import synth_clifford_bm
from qiskit.transpiler.passes.synthesis.plugin import HighLevelSynthesisPlugin


class MyCliffordSynthesisPlugin(HighLevelSynthesisPlugin):
    def run(
        self,
        high_level_object,
        coupling_map=None,
        target=None,
        qubits=None,
        **options,
    ) -> QuantumCircuit:
        if high_level_object.num_qubits <= 3:
            return synth_clifford_bm(high_level_object)
        else:
            return None

This plugin synthesizes objects of type [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford) that have
at most 3 qubits, using the `synth_clifford_bm` method.

Now, we expose the plugin by adding an entry point in our Python package metadata.
Here, we assume that the class we defined is exposed in a module called `my_qiskit_plugin`, for example by being imported in the `__init__.py` file of the `my_qiskit_plugin` module.
We edit the `pyproject.toml`, `setup.cfg`, or `setup.py` file of our package:

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.synthesis"]
    "clifford.my_clifford_synthesis" = "my_qiskit_plugin:MyCliffordSynthesisPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.synthesis =
        clifford.my_clifford_synthesis = my_qiskit_plugin:MyCliffordSynthesisPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.synthesis': [
                'clifford.my_clifford_synthesis = my_qiskit_plugin:MyCliffordSynthesisPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

The `name` consists of two parts separated by a dot (`.`):
- The name of the type of [Operation](/docs/api/qiskit/qiskit.circuit.Operation) that the plugin synthesizes (in this case, `clifford`). Note that this string corresponds to the [`name`](/docs/api/qiskit/qiskit.circuit.Operation#name) attribute of the Operation class, and not the name of the class itself.
- The name of the plugin (in this case, `special`).

As before, if your project uses `setup.cfg` or `setup.py` instead of `pyproject.toml`, see the [setuptools documentation](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) for how to adapt these lines for your situation.

To check that your plugin is successfully detected by Qiskit, install your plugin package and follow the instructions at [Transpiler plugins](transpiler-plugins#list-available-high-level-synthesis-plugins) for listing installed plugins, and ensure that your plugin appears in the list:

In [7]:
from qiskit.transpiler.passes.synthesis import (
    high_level_synthesis_plugin_names,
)

high_level_synthesis_plugin_names("clifford")

['ag', 'bm', 'default', 'greedy', 'layers', 'lnn', 'rb_default']

لو كانت إضافتنا في المثال مثبَّتة، لظهر الاسم `my_unitary_synthesis` في هذه القائمة.

لاستيعاب إضافات توليف الوحدانيات التي تكشف عن خيارات متعددة،
تتضمن واجهة الإضافة خياراً يتيح للمستخدمين تمرير قاموس تكوين حر الصياغة. سيُمرَّر هذا القاموس إلى طريقة `run`
عبر وسيط الكلمة المفتاحية `options`. إذا كانت إضافتك تمتلك هذه الخيارات التكوينية، يجب عليك توثيقها بوضوح.

## مثال: إنشاء إضافة للتوليف عالي المستوى

في هذا المثال، سننشئ إضافة للتوليف عالي المستوى تستخدم ببساطة الدالة المدمجة [synth_clifford_bm](https://docs.quantum.ibm.com/api/qiskit/synthesis#synth_clifford_bm) لتوليف عامل Clifford.

تُعرِّف فئة [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) الواجهة والعقد الخاص بإضافات التوليف عالي المستوى. الطريقة الرئيسية هي [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin#run).
الوسيط الموضعي `high_level_object` هو [Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation) يُمثّل الكائن "عالي المستوى" المراد توليفه. على سبيل المثال، يمكن أن يكون
[LinearFunction](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) أو
[Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford).
وسيطات الكلمات المفتاحية التالية متاحة:
- يُحدِّد `target` الـ Backend المستهدف، مما يتيح للإضافة الوصول إلى جميع المعلومات الخاصة بالهدف،
كخريطة الاقتران، ومجموعة Gates المدعومة، وما إلى ذلك
- يُحدِّد `coupling_map` خريطة الاقتران فقط، ويُستخدم فقط عندما لا يكون `target` مُحدَّداً.
- يُحدِّد `qubits` قائمة Qubits التي يُعرَّف عليها
الكائن عالي المستوى، في حالة إجراء التوليف على الدائرة المادية.
تعني القيمة ``None`` أن التخطيط لم يُختَر بعد ولم يتحدد بعد أي Qubits المادية في الهدف أو خريطة الاقتران التي ستعمل عليها هذه العملية.
- `options`، قاموس تكوين حر الصياغة لخيارات خاصة بالإضافة. إذا كانت إضافتك تمتلك هذه الخيارات التكوينية
فيجب عليك توثيقها بوضوح.

تُعيد طريقة `run` دائرة [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit)
تُمثّل الدائرة المُولَّفة من ذلك الكائن عالي المستوى.
يُسمح أيضاً بإعادة `None`، مما يشير إلى أن الإضافة غير قادرة على توليف الكائن عالي المستوى المعطى.
يتولى التوليف الفعلي للكائنات عالية المستوى تمريرة الـ Transpiler
[HighLevelSynthesis](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.HighLevelSynthesis).

إلى جانب طريقة `run`، ثمّة عدد من طرق الخصائص التي يجب تعريفها.
راجع [HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) للاطلاع على توثيق جميع الخصائص المطلوبة.

لنُعرِّف الآن فئتنا الفرعية من HighLevelSynthesisPlugin: